# Bài thực hành số 2 — Thu thập, lưu trữ và tiền xử lý dữ liệu IoT
**IoT và Ứng dụng (INT14149)** · Trần Mạnh Khiêm · B23DCCN437 · Nhóm 04 tổ 2 — *nối tiếp Bài thí nghiệm số 1*

Pipeline: **ESP32 trên Wokwi** (HC-SR04, DHT22, LED) → MQTT → **ThingsBoard Cloud** → REST API → **Google Colab** (notebook này: thu thập, kiểm tra, tiền xử lý, đo độ trễ) → **InfluxDB Cloud** (time-series DB).

Trước khi chạy: (1) mở project Wokwi của bài 1 (`firmware/esp32_wokwi`) và bấm Start để thiết bị gửi dữ liệu; (2) tạo API key ThingsBoard (*Account → Security → API keys*); (3) *Runtime → Run all*, nhập API key khi được hỏi.

In [ ]:
print("Bat dau bai thuc hanh IoT so 2")

import sys
print("Phien ban Python:", sys.version)

In [ ]:
%pip install requests pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime

print("Da import thu vien thanh cong!")
print("Pandas version:", pd.__version__)

## 1. Cấu hình ThingsBoard

In [ ]:
DEVICE_ID = "dff594d0-bfb7-11f1-b930-ad60e0953111"

print("Device ID:", DEVICE_ID)

In [ ]:
import requests
import pandas as pd
import time

THINGSBOARD_URL = "https://thingsboard.cloud"
DEVICE_ID = "dff594d0-bfb7-11f1-b930-ad60e0953111"

print("Da khoi tao thong tin ThingsBoard")
print("Device ID:", DEVICE_ID)

## 2. Nhập API key (getpass, không hiện giá trị) và kiểm tra kết nối

In [ ]:
from getpass import getpass

API_KEY = getpass("Nhap API key ThingsBoard: ")
print("Da nhap API key, khong hien thi gia tri.")

In [ ]:
url = f"{THINGSBOARD_URL}/api/auth/user"

headers = {
    "X-Authorization": f"ApiKey {API_KEY}",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers, timeout=20)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    print("Ket noi ThingsBoard API thanh cong!")
    print("Thong tin tai khoan:", response.json())
elif response.status_code == 401:
    print("API key khong hop le, het han hoac khong duoc chap nhan.")
elif response.status_code == 403:
    print("Da xac thuc nhung khong du quyen truy cap.")
else:
    print("Phan hoi:", response.text[:300])

## 3. Thu thập dữ liệu telemetry qua REST API
Lấy bản tin mới nhất, sau đó lấy toàn bộ dữ liệu 30 phút gần nhất (`startTs`, `endTs`, `agg=NONE`) và ghép các key theo timestamp.

In [ ]:
# Lay cac truong telemetry cua ESP32
keys = "distance_cm,rssi,sequence,uptime_s"

url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{DEVICE_ID}/values/timeseries"

params = {
    "keys": keys,
    "limit": 10
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=20
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    telemetry = response.json()

    for key, values in telemetry.items():
        print(f"\n{key}:")
        for item in values:
            print(item)
else:
    print("Loi:", response.text[:500])

In [ ]:
# Chuyen telemetry JSON thanh bang pandas

records = []

for key, values in telemetry.items():
    for item in values:
        records.append({
            "timestamp": item["ts"],
            "key": key,
            "value": float(item["value"])
        })

df_long = pd.DataFrame(records)

# Chuyen tu dang doc sang dang rong:
# Moi hang la mot mau, moi cot la mot truong telemetry
df = df_long.pivot_table(
    index="timestamp",
    columns="key",
    values="value",
    aggfunc="last"
).reset_index()

# Chuyen timestamp Unix milliseconds thanh datetime UTC
df["timestamp"] = pd.to_datetime(
    df["timestamp"], unit="ms", utc=True
)

df = df.sort_values("timestamp").reset_index(drop=True)

print("So mau du lieu:", len(df))
display(df)

In [ ]:
import time

# Lay du lieu trong 30 phut gan nhat
end_ts = int(time.time() * 1000)
start_ts = end_ts - 30 * 60 * 1000

params = {
    "keys": "distance_cm,rssi,sequence,uptime_s",
    "startTs": start_ts,
    "endTs": end_ts,
    "limit": 500,
    "agg": "NONE"
}

url = (
    f"{THINGSBOARD_URL}/api/plugins/telemetry/"
    f"DEVICE/{DEVICE_ID}/values/timeseries"
)

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    history = response.json()

    for key, values in history.items():
        print(f"{key}: {len(values)} ban ghi")
else:
    print(response.text[:500])

In [ ]:
# Chuyen du lieu lich su thanh bang
records = []

for key, values in history.items():
    for item in values:
        records.append({
            "timestamp_ms": int(item["ts"]),
            "key": key,
            "value": float(item["value"])
        })

history_long = pd.DataFrame(records)

history_df = (
    history_long
    .pivot_table(
        index="timestamp_ms",
        columns="key",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

# Chuyen timestamp sang datetime UTC
history_df["timestamp"] = pd.to_datetime(
    history_df["timestamp_ms"],
    unit="ms",
    utc=True
)

history_df = history_df.drop(columns=["timestamp_ms"])
history_df = history_df[
    ["timestamp", "distance_cm", "rssi", "sequence", "uptime_s"]
].sort_values("timestamp").reset_index(drop=True)

print("Tong so mau sau khi ghep:", len(history_df))
print("\nSo gia tri thieu tren tung cot:")
print(history_df.isna().sum())

display(history_df.head(10))

## 4. Kiểm tra dữ liệu và phát hiện ngoại lệ
Thống kê mô tả, IQR cho từng cột, xem phân bố giá trị khoảng cách, kiểm tra miền đo của HC-SR04 (2–400 cm).

In [ ]:
# Cac cot can kiem tra ngoai le
numeric_cols = [
    "distance_cm",
    "rssi",
    "sequence",
    "uptime_s"
]

# Thong ke mo ta
display(history_df[numeric_cols].describe().T)

# Phat hien ngoai le bang IQR
outlier_summary = []

for col in numeric_cols:
    q1 = history_df[col].quantile(0.25)
    q3 = history_df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (
        (history_df[col] < lower) |
        (history_df[col] > upper)
    )

    outlier_summary.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum())
    })

outlier_df = pd.DataFrame(outlier_summary)

display(outlier_df)

In [ ]:
# Xem các giá trị khoảng cách khác nhau
print("Các giá trị distance_cm:")
print(sorted(history_df["distance_cm"].dropna().unique()))

# Xem những dòng bị IQR đánh dấu
q1 = history_df["distance_cm"].quantile(0.25)
q3 = history_df["distance_cm"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = history_df[
    (history_df["distance_cm"] < lower) |
    (history_df["distance_cm"] > upper)
]

print("Số dòng bị đánh dấu:", len(outliers))
display(outliers[["timestamp", "distance_cm"]].head(20))

In [ ]:
# Kiểm tra phạm vi và tần suất của từng giá trị khoảng cách
distance_counts = (
    history_df["distance_cm"]
    .value_counts()
    .sort_index()
    .rename_axis("distance_cm")
    .reset_index(name="count")
)

display(distance_counts)

print("Giá trị nhỏ nhất:", history_df["distance_cm"].min(), "cm")
print("Giá trị lớn nhất:", history_df["distance_cm"].max(), "cm")

In [ ]:
print("Thống kê khoảng cách:")
display(history_df["distance_cm"].describe())

print("\nSố lần xuất hiện của từng khoảng cách:")
display(
    history_df["distance_cm"]
    .value_counts()
    .sort_index()
    .rename_axis("distance_cm")
    .reset_index(name="count")
)

In [ ]:
numeric_cols = [
    "distance_cm", "rssi", "sequence", "uptime_s"
]

outlier_summary = []

for col in numeric_cols:
    q1 = history_df[col].quantile(0.25)
    q3 = history_df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (
        (history_df[col] < lower) |
        (history_df[col] > upper)
    )

    outlier_summary.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum())
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

In [ ]:
invalid_mask = (
    (history_df["distance_cm"] < 2) |
    (history_df["distance_cm"] > 400)
)

invalid_df = history_df.loc[
    invalid_mask,
    ["timestamp", "distance_cm"]
].copy()

print("Số mẫu ngoài phạm vi 2–400 cm:", len(invalid_df))
display(invalid_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(
    history_df["timestamp"],
    history_df["distance_cm"],
    marker=".",
    markersize=3
)

plt.title("Khoảng cách HC-SR04 theo thời gian")
plt.xlabel("Thời gian")
plt.ylabel("Khoảng cách (cm)")
plt.xticks(rotation=30)
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Tiền xử lý
Loại mẫu ngoài miền đo, tạo đặc trưng trung bình trượt 3 mẫu và độ thay đổi, chuẩn hóa Min-Max, xử lý giá trị thiếu.

In [ ]:
# Sao chép dữ liệu gốc
clean_df = history_df.copy()

# Đánh dấu khoảng cách ngoài phạm vi tham khảo 2–400 cm
invalid_mask = (
    (clean_df["distance_cm"] < 2) |
    (clean_df["distance_cm"] > 400)
)

# Lưu riêng các mẫu cần kiểm tra
invalid_df = clean_df.loc[invalid_mask].copy()

# Loại các mẫu ngoài phạm vi khỏi bộ dữ liệu xử lý
clean_df = clean_df.loc[~invalid_mask].copy()

print("Số mẫu gốc:", len(history_df))
print("Số mẫu ngoài phạm vi:", len(invalid_df))
print("Số mẫu sau lọc:", len(clean_df))

print("\nKiểm tra khoảng cách sau lọc:")
print(clean_df["distance_cm"].describe())

In [ ]:
# Sắp xếp theo thời gian
clean_df = clean_df.sort_values("timestamp").reset_index(drop=True)

# Bước 15: Trung bình trượt trên 3 mẫu
clean_df["distance_rolling_mean"] = (
    clean_df["distance_cm"]
    .rolling(window=3, min_periods=1)
    .mean()
)

# Bước 16: Độ thay đổi so với mẫu trước
clean_df["distance_delta"] = clean_df["distance_cm"].diff()

display(
    clean_df[
        [
            "timestamp",
            "distance_cm",
            "distance_rolling_mean",
            "distance_delta"
        ]
    ].head(15)
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Các trường đo và đặc trưng cần chuẩn hóa
scale_cols = [
    "distance_cm",
    "rssi",
    "distance_rolling_mean",
    "distance_delta"
]

# Chỉ chuẩn hóa các cột đang tồn tại trong clean_df
scale_cols = [c for c in scale_cols if c in clean_df.columns]

scaler = MinMaxScaler()

clean_df[[f"{c}_scaled" for c in scale_cols]] = (
    scaler.fit_transform(clean_df[scale_cols])
)

display(clean_df.head(10))

## 6. Lưu kết quả tiền xử lý ra CSV

In [ ]:
output_path = "/content/iot_telemetry_processed.csv"

clean_df.to_csv(output_path, index=False)

print("Đã lưu:", output_path)
print("Số mẫu:", len(clean_df))
print("Số cột:", len(clean_df.columns))

In [ ]:
from google.colab import files
files.download("/content/iot_telemetry_processed.csv")

In [ ]:
# Kiểm tra số dòng, số cột và dữ liệu thiếu
print("Kích thước dữ liệu:", clean_df.shape)
print("\nSố giá trị thiếu:")
print(clean_df.isnull().sum())

# Xem 5 dòng đầu
display(clean_df.head())

# Lưu dữ liệu đã xử lý
clean_df.to_csv(
    "/content/iot_telemetry_processed.csv",
    index=False
)

print("Đã lưu file iot_telemetry_processed.csv")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Đảm bảo timestamp được nhận dạng là thời gian
clean_df["timestamp"] = pd.to_datetime(clean_df["timestamp"])

# Biểu đồ 1: Khoảng cách đo được theo thời gian
plt.figure(figsize=(12, 5))
plt.plot(
    clean_df["timestamp"],
    clean_df["distance_cm"],
    label="Khoảng cách đo được"
)
plt.plot(
    clean_df["timestamp"],
    clean_df["distance_rolling_mean"],
    label="Trung bình trượt (3 mẫu)"
)
plt.xlabel("Thời gian")
plt.ylabel("Khoảng cách (cm)")
plt.title("Khoảng cách cảm biến HC-SR04 theo thời gian")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

# Biểu đồ 2: Độ thay đổi khoảng cách
plt.figure(figsize=(12, 4))
plt.plot(
    clean_df["timestamp"],
    clean_df["distance_delta"]
)
plt.xlabel("Thời gian")
plt.ylabel("Độ thay đổi (cm)")
plt.title("Độ thay đổi khoảng cách giữa hai mẫu liên tiếp")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra số lượng giá trị thiếu trước khi xử lý
print("Trước khi xử lý:")
print(clean_df.isnull().sum())

# Mẫu đầu tiên không có độ thay đổi nên gán bằng 0
clean_df["distance_delta"] = clean_df["distance_delta"].fillna(0)

# Tính lại cột chuẩn hóa sau khi xử lý giá trị thiếu
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

clean_df["distance_delta_scaled"] = scaler.fit_transform(
    clean_df[["distance_delta"]]
)

print("\nSau khi xử lý:")
print(clean_df.isnull().sum())

In [ ]:
# Lưu dữ liệu đã tiền xử lý
output_path = "/content/iot_telemetry_processed.csv"

clean_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Đã lưu file:", output_path)
print("Số dòng:", len(clean_df))
print("Số cột:", len(clean_df.columns))

# Hiển thị 5 dòng đầu tiên để kiểm tra
display(clean_df.head())

In [ ]:
from google.colab import files

files.download("/content/iot_telemetry_processed.csv")

## 7. Lưu vào cơ sở dữ liệu time-series InfluxDB Cloud
Đăng ký miễn phí tại https://cloud2.influxdata.com (bucket đầu tiên đặt tên `iot_raw`, tạo thêm bucket `iot_processed` trong *Load Data → Buckets*), sửa `INFLUX_URL`, `INFLUX_ORG` bên dưới. Schema: measurement `sensor_raw`, tag `device_id`, field `distance_cm`, `rssi` (float), `sequence`, `uptime_s` (integer); timestamp = timestamp ThingsBoard nên ghi lặp chỉ ghi đè.

In [ ]:
%pip install -q influxdb-client
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

INFLUX_URL = "https://us-east-1-1.aws.cloud2.influxdata.com"   # sửa theo region của bạn
INFLUX_ORG = "TEN_ORGANIZATION"                                # sửa
db = InfluxDBClient(url=INFLUX_URL, token=getpass("Nhap InfluxDB token: "), org=INFLUX_ORG, timeout=30000)
w = db.write_api(write_options=SYNCHRONOUS)

pts = [Point("sensor_raw").tag("device_id", DEVICE_ID)
       .field("distance_cm", float(r.distance_cm)).field("rssi", float(r.rssi))
       .field("sequence", int(r.sequence)).field("uptime_s", int(r.uptime_s))
       .time(r.timestamp.to_pydatetime(), WritePrecision.MS) for r in clean_df.itertuples()]
t0 = time.time()
w.write("iot_raw", record=pts)
print("Ghi", len(pts), "diem vao iot_raw:", round((time.time() - t0) * 1000), "ms")

proc = clean_df.set_index("timestamp").drop(columns=["sequence", "uptime_s"])
proc["device_id"] = DEVICE_ID
w.write("iot_processed", record=proc, data_frame_measurement_name="sensor_processed",
        data_frame_tag_columns=["device_id"], write_precision=WritePrecision.MS)

# Đọc lại theo khoảng thời gian bằng Flux
flux = f'''from(bucket: "iot_raw") |> range(start: -2h)
  |> filter(fn: (r) => r._measurement == "sensor_raw" and r._field == "distance_cm")'''
t0 = time.time()
back = db.query_api().query_data_frame(flux)
print("Doc lai", len(back), "diem trong", round((time.time() - t0) * 1000), "ms")

## 8. Dashboard trên Colab (Plotly)

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# Chuẩn bị dữ liệu
dashboard_df = clean_df.copy()
dashboard_df["timestamp"] = pd.to_datetime(dashboard_df["timestamp"])

# Các chỉ số tổng quan
print("=== TỔNG QUAN DỮ LIỆU IoT ===")
print("Tổng số mẫu:", len(dashboard_df))
print("Khoảng cách trung bình:", round(dashboard_df["distance_cm"].mean(), 2), "cm")
print("RSSI trung bình:", round(dashboard_df["rssi"].mean(), 2), "dBm")
print("Khoảng cách nhỏ nhất:", round(dashboard_df["distance_cm"].min(), 2), "cm")
print("Khoảng cách lớn nhất:", round(dashboard_df["distance_cm"].max(), 2), "cm")

# Biểu đồ khoảng cách theo thời gian
fig = px.line(
    dashboard_df,
    x="timestamp",
    y=["distance_cm", "distance_rolling_mean"],
    title="Dashboard IoT - Khoảng cách cảm biến HC-SR04",
    labels={
        "timestamp": "Thời gian",
        "value": "Khoảng cách (cm)",
        "variable": "Loại dữ liệu"
    }
)

fig.show()

In [ ]:
import requests
import pandas as pd
from IPython.display import display

url = (
    f"{THINGSBOARD_URL}/api/plugins/telemetry/"
    f"DEVICE/{DEVICE_ID}/values/timeseries"
)

params = {
    "keys": "distance_cm,rssi,sequence,uptime_s",
    "limit": 100
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()
data = response.json()

latest_data = {}

for key, values in data.items():
    for item in values:
        ts = item["ts"]
        latest_data.setdefault(ts, {"timestamp": ts})
        latest_data[ts][key] = float(item["value"])

latest_df = pd.DataFrame(list(latest_data.values()))

if not latest_df.empty:
    latest_df["timestamp"] = pd.to_datetime(
        latest_df["timestamp"],
        unit="ms",
        utc=True
    )

    latest_df = latest_df.sort_values("timestamp")

    print("Số mẫu lấy được:", len(latest_df))
    display(latest_df.tail(10))
else:
    print("Chưa có dữ liệu telemetry.")

In [ ]:
import plotly.express as px

if not latest_df.empty:
    fig = px.line(
        latest_df,
        x="timestamp",
        y="distance_cm",
        markers=True,
        title="Khoảng cách cảm biến HC-SR04 - Dữ liệu mới nhất",
        labels={
            "timestamp": "Thời gian",
            "distance_cm": "Khoảng cách (cm)"
        }
    )

    fig.show()
else:
    print("Không có dữ liệu để vẽ biểu đồ.")

In [ ]:
import time
import requests
import pandas as pd

end_ts = int(time.time() * 1000)
start_ts = end_ts - 30 * 60 * 1000

params = {
    "keys": "distance_cm,rssi,sequence,uptime_s",
    "startTs": start_ts,
    "endTs": end_ts,
    "interval": 1000,
    "limit": 1000,
    "agg": "NONE"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()
data = response.json()

rows = {}

for key, values in data.items():
    for item in values:
        ts = item["ts"]
        rows.setdefault(ts, {"timestamp": ts})
        rows[ts][key] = float(item["value"])

latest_df = pd.DataFrame(list(rows.values()))

if not latest_df.empty:
    latest_df["timestamp"] = pd.to_datetime(
        latest_df["timestamp"], unit="ms", utc=True
    )
    latest_df = latest_df.sort_values("timestamp").reset_index(drop=True)

    print("Số mẫu lịch sử:", len(latest_df))
    display(latest_df.head())
else:
    print("Không có dữ liệu trong khoảng thời gian đã chọn.")

In [ ]:
import plotly.express as px
from IPython.display import display

# Biểu đồ khoảng cách theo thời gian
fig1 = px.line(
    latest_df,
    x="timestamp",
    y="distance_cm",
    title="Khoảng cách HC-SR04 theo thời gian",
    labels={
        "timestamp": "Thời gian",
        "distance_cm": "Khoảng cách (cm)"
    }
)
fig1.show()

# Biểu đồ cường độ tín hiệu Wi-Fi
fig2 = px.line(
    latest_df,
    x="timestamp",
    y="rssi",
    title="Cường độ tín hiệu Wi-Fi theo thời gian",
    labels={
        "timestamp": "Thời gian",
        "rssi": "RSSI (dBm)"
    }
)
fig2.show()

# Các chỉ số tổng quan
print("Tổng số mẫu:", len(latest_df))
print("Khoảng cách trung bình:",
      round(latest_df["distance_cm"].mean(), 2), "cm")
print("Khoảng cách nhỏ nhất:",
      round(latest_df["distance_cm"].min(), 2), "cm")
print("Khoảng cách lớn nhất:",
      round(latest_df["distance_cm"].max(), 2), "cm")
print("RSSI trung bình:",
      round(latest_df["rssi"].mean(), 2), "dBm")

## 9. Đo độ trễ truy vấn REST API

In [ ]:
import time
import requests
import pandas as pd

# Đo thời gian gửi yêu cầu và nhận dữ liệu
start = time.perf_counter()

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

end = time.perf_counter()

response.raise_for_status()

latency_ms = (end - start) * 1000

print("HTTP status:", response.status_code)
print(f"Thời gian truy vấn: {latency_ms:.2f} ms")
print(f"Thời gian truy vấn: {latency_ms / 1000:.3f} giây")

In [ ]:
import time
import requests
import pandas as pd

latencies = []

for i in range(10):
    start = time.perf_counter()

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    end = time.perf_counter()
    response.raise_for_status()

    latency_ms = (end - start) * 1000
    latencies.append(latency_ms)

    print(f"Lần {i + 1}: {latency_ms:.2f} ms")

print("\n--- KẾT QUẢ TỔNG HỢP ---")
print(f"Trung bình: {sum(latencies) / len(latencies):.2f} ms")
print(f"Nhỏ nhất:   {min(latencies):.2f} ms")
print(f"Lớn nhất:   {max(latencies):.2f} ms")
print(f"Độ lệch chuẩn: {pd.Series(latencies).std():.2f} ms")

In [ ]:
import pandas as pd
from google.colab import files

latency_df = pd.DataFrame({
    "lan_do": range(1, len(latencies) + 1),
    "do_tre_ms": latencies
})

latency_df.to_csv(
    "/content/latency_results.csv",
    index=False,
    encoding="utf-8-sig"
)

display(latency_df)

files.download("/content/latency_results.csv")